# Multicity Weather Prediction using LSTM
Predict weather of one city using data from multiple cities.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

## Load Dataset
Expected format:
Date, Temp_City1, Temp_City2, Temp_City3, Target_Temp

In [ ]:
# Synthetic dataset (replace with real weather data)
dates = pd.date_range(start='2020-01-01', periods=500)

data = pd.DataFrame({
    'Date': dates,
    'City1': np.sin(np.arange(500)/10) * 10 + 25,
    'City2': np.sin(np.arange(500)/11) * 8 + 24,
    'City3': np.sin(np.arange(500)/9) * 9 + 26,
    'Target': np.sin(np.arange(500)/10.5) * 10 + 25
})

data.set_index('Date', inplace=True)
data.head()

## Visualization

In [ ]:
data.plot(figsize=(10,5))
plt.title('Multicity Temperature Data')
plt.show()

## Preprocessing

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

In [ ]:
def create_sequences(data, seq_length=30):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, :-1])  # all cities except target
        y.append(data[i, -1])  # target city
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data)

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(X_train.shape)

## Build LSTM Model

In [ ]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(32))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')
model.summary()

## Train Model

In [ ]:
model.fit(X_train, y_train, epochs=20, batch_size=32)

## Predictions

In [ ]:
predictions = model.predict(X_test)

## Plot Results

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(y_test, label='Actual')
plt.plot(predictions, label='Predicted')
plt.legend()
plt.title('Weather Prediction (Target City)')
plt.show()